In [1]:
# ============================================================
# MD13 — DEBERTA-V3-BASE + SOFTENED WEIGHTED CROSS-ENTROPY
#
# FIXES APPLIED:
#   FIX 1:
#     Weighted cross-entropy is computed in float32 so fp16 logits and
#     float32 class weights do not cause a dtype mismatch.
#
#   FIX 2:
#     DeBERTa-v3-base checkpoint parameters can load in float16.
#     AMP/GradScaler requires FP32 master trainable parameters; otherwise
#     training can fail with:
#         ValueError: Attempting to unscale FP16 gradients.
#     Therefore the model is explicitly converted to float32 BEFORE
#     Trainer/AMP setup. fp16=True is still used for mixed-precision
#     forward/backward computation.
#
# Research question:
#   Can a stronger single 512-token encoder outperform the
#   current best selector (MD6 DistilRoBERTa)?
#
# Model change:
#   DistilRoBERTa / RoBERTa-base
#       ->
#   microsoft/deberta-v3-base
#
# Held fixed from the successful MD6-style setup:
#   - same train/validation files
#   - same observable selector inputs
#   - same paired-input formatting
#   - same left truncation of conversation only
#   - max_length = 512
#   - inverse-frequency class weights ** 0.75
#   - learning_rate = 2e-5
#   - effective global batch = 16
#   - epochs = 3
#   - weight_decay = 0.01
#   - warmup_steps = 280
#   - seed = 42
#
# Memory-safe 2xT4 setup:
#   per-device batch = 2
#   GPUs = 2
#   gradient accumulation = 4
#   effective batch = 2 * 2 * 4 = 16
#
# Primary metric: Macro-F1
# Secondary metric: Accuracy
# Test set: NOT USED
# ============================================================

import gc
import json
import math
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 512
WEIGHT_ALPHA = 0.75
LEARNING_RATE = 2e-5

PER_DEVICE_TRAIN_BATCH = 2
GRAD_ACCUM = 4
EXPECTED_GPUS = 2
EFFECTIVE_BATCH = (
    PER_DEVICE_TRAIN_BATCH
    * EXPECTED_GPUS
    * GRAD_ACCUM
)
EPOCHS = 3

TRAIN_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl"
)
VAL_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl"
)

OUTPUT_DIR = Path(
    "/kaggle/working/md13_debertav3_base_weighted_alpha075"
)
RESULT_DIR = Path(
    "/kaggle/working/md13_results"
)
RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LABELS = [
    "generic",
    "probing",
    "focus",
    "telling",
]
LABEL2ID = {
    label: i
    for i, label
    in enumerate(LABELS)
}
ID2LABEL = {
    i: label
    for label, i
    in LABEL2ID.items()
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 78)
print("MD13 CONFIGURATION")
print("=" * 78)
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Weight alpha:", WEIGHT_ALPHA)
print("Learning rate:", LEARNING_RATE)
print("Per-device batch:", PER_DEVICE_TRAIN_BATCH)
print("Gradient accumulation:", GRAD_ACCUM)
print("Expected GPUs:", EXPECTED_GPUS)
print("Effective global batch:", EFFECTIVE_BATCH)
print("Epochs:", EPOCHS)
print("Detected GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "GPU is required."

assert torch.cuda.device_count() == EXPECTED_GPUS, (
    "MD13 is defined for Kaggle 2xT4. "
    f"Detected {torch.cuda.device_count()} GPU(s)."
)

assert EFFECTIVE_BATCH == 16

def load_jsonl(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

assert len(train_rows) == 14905
assert len(val_rows) == 1850

required = {
    "example_id",
    "qid",
    "problem",
    "history",
    "target_move",
}

for name, rows in [
    ("train", train_rows),
    ("validation", val_rows),
]:
    assert required.issubset(
        rows[0].keys()
    ), f"{name} schema mismatch"

    bad = [
        r
        for r in rows
        if r["target_move"]
        not in LABEL2ID
    ]

    assert not bad, (
        f"{name}: invalid labels found"
    )

print("Train examples:", len(train_rows))
print("Validation examples:", len(val_rows))
print("Test set used: False")


MD13 CONFIGURATION
Model: microsoft/deberta-v3-base
Max length: 512
Weight alpha: 0.75
Learning rate: 2e-05
Per-device batch: 2
Gradient accumulation: 4
Expected GPUs: 2
Effective global batch: 16
Epochs: 3
Detected GPU count: 2
Train examples: 14905
Validation examples: 1850
Test set used: False


In [2]:
# ============================================================
# INPUT FORMAT + TOKENIZATION + FIXED DISTIL LENGTH DIAGNOSTIC
# ============================================================

def format_history(history):
    if not history:
        return "[No previous conversation]"

    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )

def build_problem_text(example):
    return (
        "Problem:\n"
        + str(example["problem"])
    )

def build_conversation_text(example):
    return (
        "Conversation:\n"
        + format_history(
            example["history"]
        )
        + "\n\nNext teacher pedagogical move:"
    )

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)
tokenizer.truncation_side = "left"

def prepare_rows(rows):
    prepared = []

    for ex in rows:
        prepared.append({
            "example_id": ex["example_id"],
            "qid": ex["qid"],
            "problem_text": (
                build_problem_text(ex)
            ),
            "conversation_text": (
                build_conversation_text(ex)
            ),
            "labels": LABEL2ID[
                ex["target_move"]
            ],
        })

    return prepared

train_prepared = prepare_rows(
    train_rows
)
val_prepared = prepare_rows(
    val_rows
)

# ------------------------------------------------------------
# Keep the historical 1507/343 DistilRoBERTa-defined split
# purely as a diagnostic, so MD13 can be compared directly
# with MD6/MD8/MD9/MD10/MD11/MD12.
# ------------------------------------------------------------

distil_tokenizer = (
    AutoTokenizer.from_pretrained(
        "distilbert/distilroberta-base"
    )
)
distil_tokenizer.truncation_side = "left"

val_original_lengths = []

for ex in val_rows:
    encoded = distil_tokenizer(
        build_problem_text(ex),
        build_conversation_text(ex),
        add_special_tokens=True,
        truncation=False,
    )

    val_original_lengths.append(
        len(encoded["input_ids"])
    )

val_original_lengths = np.asarray(
    val_original_lengths
)

short_mask = (
    val_original_lengths <= 512
)
long_mask = (
    val_original_lengths > 512
)

print(
    "Validation <=512:",
    int(short_mask.sum()),
)
print(
    "Validation >512:",
    int(long_mask.sum()),
)

assert int(short_mask.sum()) == 1507
assert int(long_mask.sum()) == 343

train_ds = Dataset.from_list(
    train_prepared
)
val_ds = Dataset.from_list(
    val_prepared
)

def tokenize_batch(batch):
    return tokenizer(
        batch["problem_text"],
        batch["conversation_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
        "example_id",
        "qid",
    ],
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "problem_text",
        "conversation_text",
        "example_id",
        "qid",
    ],
)

assert max(
    len(x)
    for x in train_tok["input_ids"]
) <= MAX_LENGTH

assert max(
    len(x)
    for x in val_tok["input_ids"]
) <= MAX_LENGTH

data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer
    )
)

print("Tokenization: PASSED")


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


Validation <=512: 1507
Validation >512: 343


Map:   0%|          | 0/14905 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]

Tokenization: PASSED


In [3]:
# ============================================================
# ALPHA=0.75 CLASS WEIGHTS
# ============================================================

train_counts = Counter(
    r["target_move"]
    for r in train_rows
)

n = len(train_rows)
k = len(LABELS)

raw_class_weights = np.asarray(
    [
        n / (
            k
            * train_counts[label]
        )
        for label in LABELS
    ],
    dtype=np.float32,
)

class_weights = np.power(
    raw_class_weights,
    WEIGHT_ALPHA,
).astype(np.float32)

print("=" * 78)
print("MD13 CLASS WEIGHTS")
print("=" * 78)

for label, raw_w, tuned_w in zip(
    LABELS,
    raw_class_weights,
    class_weights,
):
    print(
        f"{label:10s} "
        f"count={train_counts[label]:4d} "
        f"raw={raw_w:.6f} "
        f"alpha075={tuned_w:.6f}"
    )


MD13 CLASS WEIGHTS
generic    count=3579 raw=1.041143 alpha075=1.030701
probing    count=3413 raw=1.091781 alpha075=1.068075
focus      count=5443 raw=0.684595 alpha075=0.752619
telling    count=2470 raw=1.508603 alpha075=1.361229


In [4]:
# ============================================================
# MODEL + WEIGHTED TRAINER + TRAINING
# ============================================================

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        label2id=LABEL2ID,
        id2label=ID2LABEL,
    )
)

# ------------------------------------------------------------
# CRITICAL AMP FIX:
# Keep FP32 master trainable parameters.
#
# The DeBERTa-v3-base checkpoint may materialize parameters in
# float16. GradScaler cannot safely unscale gradients when the
# optimizer itself owns FP16 parameters. Converting the model to
# float32 here gives AMP the normal setup:
#
#   master parameters / optimizer states -> float32
#   autocast forward computation         -> float16 where safe
#   weighted CE                          -> float32
# ------------------------------------------------------------
model = model.float()

parameter_dtypes = sorted(
    {str(p.dtype) for p in model.parameters()}
)

print("Model parameter dtypes before Trainer:", parameter_dtypes)

assert parameter_dtypes == ["torch.float32"], (
    "All trainable/model parameters must be float32 before "
    "Trainer initializes mixed-precision training. "
    f"Found: {parameter_dtypes}"
)

class WeightedTrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):
        super().__init__(
            *args,
            **kwargs,
        )

        self.class_weights_tensor = (
            torch.tensor(
                class_weights,
                dtype=torch.float32,
            )
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop(
            "labels"
        )

        outputs = model(
            **inputs
        )

        logits = outputs.logits

        # DeBERTa is trained with fp16=True, so logits can be float16.
        # CrossEntropy requires the class-weight tensor to have the same
        # floating dtype as the input logits. Compute the loss in float32
        # for numerical stability while preserving gradient flow.
        weights = self.class_weights_tensor.to(
            device=logits.device,
            dtype=torch.float32,
        )

        loss = F.cross_entropy(
            logits.float(),
            labels,
            weight=weights,
        )

        if return_outputs:
            return loss, outputs

        return loss

def compute_metrics(
    eval_prediction
):
    logits, labels = (
        eval_prediction
    )

    pred = np.argmax(
        logits,
        axis=1,
    )

    return {
        "accuracy": accuracy_score(
            labels,
            pred,
        ),
        "macro_f1": f1_score(
            labels,
            pred,
            labels=list(
                range(len(LABELS))
            ),
            average="macro",
            zero_division=0,
        ),
    }

steps_per_epoch = math.ceil(
    len(train_rows)
    / EFFECTIVE_BATCH
)

print(
    "Expected optimizer steps/epoch:",
    steps_per_epoch,
)
print(
    "Expected total optimizer steps:",
    steps_per_epoch * EPOCHS,
)

assert steps_per_epoch == 932

training_args = TrainingArguments(
    output_dir=str(
        OUTPUT_DIR
    ),

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=(
        PER_DEVICE_TRAIN_BATCH
    ),
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=(
        GRAD_ACCUM
    ),

    num_train_epochs=EPOCHS,

    weight_decay=0.01,
    warmup_steps=280,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    load_best_model_at_end=False,

    fp16=True,

    seed=SEED,
    data_seed=SEED,

    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=train_tok,
    eval_dataset=val_tok,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print(
    "Trainer GPU count:",
    trainer.args.n_gpu,
)

assert (
    trainer.args.n_gpu
    == EXPECTED_GPUS
)

# Final pre-flight check: optimizer-facing parameters must remain FP32.
trainer_param_dtypes = sorted(
    {str(p.dtype) for p in trainer.model.parameters()}
)
print("Trainer model parameter dtypes:", trainer_param_dtypes)

assert trainer_param_dtypes == ["torch.float32"], (
    "Trainer received non-FP32 master parameters. "
    f"Found: {trainer_param_dtypes}"
)

print(
    "\n"
    + "=" * 78
)
print(
    "STARTING MD13 TRAINING"
)
print(
    "=" * 78
)

trainer.train()

history_path = (
    RESULT_DIR
    / "md13_log_history.json"
)

with history_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        trainer.state.log_history,
        f,
        indent=2,
    )

print(
    "\n"
    + "=" * 78
)
print(
    "EPOCH VALIDATION HISTORY"
)
print(
    "=" * 78
)

for row in trainer.state.log_history:
    if "eval_macro_f1" in row:
        print(
            f"Epoch {row.get('epoch')}: "
            f"accuracy="
            f"{row.get('eval_accuracy'):.6f} "
            f"macro_f1="
            f"{row.get('eval_macro_f1'):.6f} "
            f"val_loss="
            f"{row.get('eval_loss'):.6f}"
        )

print(
    "\nAll 3 epoch checkpoints saved."
)
print(
    "Best checkpoint will be selected "
    "manually by validation Macro-F1."
)


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Model parameter dtypes before Trainer: ['torch.float32']
Expected optimizer steps/epoch: 932
Expected total optimizer steps: 2796


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,5.070402,1.282037,0.448108,0.396518
2,4.844860,1.222472,0.395676,0.407369
3,4.730654,1.223421,0.422162,0.425792


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


EPOCH VALIDATION HISTORY
Epoch 1.0: accuracy=0.448108 macro_f1=0.396518 val_loss=1.282037
Epoch 2.0: accuracy=0.395676 macro_f1=0.407369 val_loss=1.222472
Epoch 3.0: accuracy=0.422162 macro_f1=0.425792 val_loss=1.223421

All 3 epoch checkpoints saved.
Best checkpoint will be selected manually by validation Macro-F1.


In [5]:
# ===========================================================
# EVALUATE ALL CHECKPOINTS + SAVE BEST MODEL
# ==========================================================

def softmax_np(logits):
    z = (
        logits
        - logits.max(
            axis=1,
            keepdims=True,
        )
    )

    ez = np.exp(z)

    return (
        ez
        / ez.sum(
            axis=1,
            keepdims=True,
        )
    )

def subset_metrics(
    true_ids,
    pred_ids,
    mask,
):
    y_true = true_ids[mask]
    y_pred = pred_ids[mask]

    return {
        "n": int(mask.sum()),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(
                    range(
                        len(LABELS)
                    )
                ),
                average="macro",
                zero_division=0,
            )
        ),
    }

checkpoint_dirs = sorted(
    OUTPUT_DIR.glob(
        "checkpoint-*"
    ),
    key=lambda p: int(
        p.name.split("-")[-1]
    ),
)

assert len(
    checkpoint_dirs
) == 3, (
    "Expected 3 checkpoints, "
    f"found {len(checkpoint_dirs)}"
)

all_results = []

for epoch_idx, ckpt in enumerate(
    checkpoint_dirs,
    start=1,
):
    print(
        "\n"
        + "-" * 78
    )
    print(
        f"EPOCH {epoch_idx}: {ckpt}"
    )
    print(
        "-" * 78
    )

    ckpt_model = (
        AutoModelForSequenceClassification
        .from_pretrained(
            ckpt
        )
        .float()
    )

    eval_args = TrainingArguments(
        output_dir=str(
            RESULT_DIR
            / f"eval_epoch{epoch_idx}"
        ),

        per_device_eval_batch_size=4,

        fp16=True,

        seed=SEED,
        data_seed=SEED,

        report_to="none",
    )

    eval_trainer = WeightedTrainer(
        model=ckpt_model,
        args=eval_args,

        eval_dataset=val_tok,

        processing_class=tokenizer,
        data_collator=data_collator,

        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    pred_output = (
        eval_trainer.predict(
            val_tok
        )
    )

    logits = (
        pred_output.predictions
    )

    true_ids = (
        pred_output.label_ids
    )

    pred_ids = np.argmax(
        logits,
        axis=1,
    )

    probs = softmax_np(
        logits
    )

    overall_accuracy = float(
        accuracy_score(
            true_ids,
            pred_ids,
        )
    )

    overall_macro_f1 = float(
        f1_score(
            true_ids,
            pred_ids,
            labels=list(
                range(
                    len(LABELS)
                )
            ),
            average="macro",
            zero_division=0,
        )
    )

    per_class_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=list(
            range(
                len(LABELS)
            )
        ),
        average=None,
        zero_division=0,
    )

    report = (
        classification_report(
            true_ids,
            pred_ids,
            labels=list(
                range(
                    len(LABELS)
                )
            ),
            target_names=LABELS,
            digits=6,
            zero_division=0,
            output_dict=True,
        )
    )

    cm = confusion_matrix(
        true_ids,
        pred_ids,
        labels=list(
            range(
                len(LABELS)
            )
        ),
    ).tolist()

    short_result = subset_metrics(
        true_ids,
        pred_ids,
        short_mask,
    )

    long_result = subset_metrics(
        true_ids,
        pred_ids,
        long_mask,
    )

    result = {
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),

        "accuracy": (
            overall_accuracy
        ),
        "macro_f1": (
            overall_macro_f1
        ),

        "per_class_f1": {
            label: float(score)
            for label, score
            in zip(
                LABELS,
                per_class_f1,
            )
        },

        "classification_report": (
            report
        ),
        "confusion_matrix": cm,

        "le512": short_result,
        "gt512": long_result,
    }

    all_results.append(
        result
    )

    print(
        "Accuracy:",
        overall_accuracy,
    )
    print(
        "Macro-F1:",
        overall_macro_f1,
    )
    print(
        "<=512 Macro-F1:",
        short_result["macro_f1"],
    )
    print(
        ">512 Macro-F1:",
        long_result["macro_f1"],
    )
    print(
        "Per-class F1:",
        result["per_class_f1"],
    )

    pred_path = (
        RESULT_DIR
        / (
            f"md13_epoch"
            f"{epoch_idx}_predictions.jsonl"
        )
    )

    with pred_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        for i, (
            true_id,
            pred_id,
            probability_row,
        ) in enumerate(
            zip(
                true_ids,
                pred_ids,
                probs,
            )
        ):
            record = {
                "example_id": (
                    val_rows[i][
                        "example_id"
                    ]
                ),
                "qid": (
                    val_rows[i][
                        "qid"
                    ]
                ),
                "true_move": (
                    ID2LABEL[
                        int(true_id)
                    ]
                ),
                "predicted_move": (
                    ID2LABEL[
                        int(pred_id)
                    ]
                ),
                "original_distil_token_length": int(
                    val_original_lengths[i]
                ),
                "probabilities": {
                    label: float(
                        probability_row[j]
                    )
                    for j, label
                    in enumerate(
                        LABELS
                    )
                },
            }

            f.write(
                json.dumps(
                    record
                )
                + "\n"
            )

    del eval_trainer
    del ckpt_model

    gc.collect()
    torch.cuda.empty_cache()

best_result = max(
    all_results,
    key=lambda x: x["macro_f1"],
)

summary = {
    "experiment": (
        "MD13_DeBERTaV3Base_"
        "weighted_CE_alpha075"
    ),

    "research_question": (
        "Can DeBERTa-v3-base with "
        "alpha=0.75 weighted cross-entropy "
        "outperform the current best "
        "single model MD6?"
    ),

    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "weight_alpha": WEIGHT_ALPHA,
    "learning_rate": LEARNING_RATE,
    "effective_global_batch": (
        EFFECTIVE_BATCH
    ),
    "epochs": EPOCHS,
    "test_used": False,

    "class_weights": {
        label: float(w)
        for label, w
        in zip(
            LABELS,
            class_weights,
        )
    },

    "baseline_md6_epoch2": {
        "accuracy": (
            0.5016216216216216
        ),
        "macro_f1": (
            0.487684572655523
        ),
    },

    "baseline_md11_epoch3": {
        "accuracy": (
            0.5021621621621621
        ),
        "macro_f1": (
            0.48428708496304046
        ),
    },

    "baseline_md12_epoch3": {
        "accuracy": (
            0.5005405405405405
        ),
        "macro_f1": (
            0.4817452965264059
        ),
    },

    "epochs_results": all_results,

    "best_epoch": (
        best_result["epoch"]
    ),
    "best_accuracy": (
        best_result["accuracy"]
    ),
    "best_macro_f1": (
        best_result["macro_f1"]
    ),
}

summary_path = (
    RESULT_DIR
    / "md13_summary.json"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        summary,
        f,
        indent=2,
    )

best_checkpoint = Path(
    best_result["checkpoint"]
)

best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        best_checkpoint
    )
    .float()
)

best_model_dir = (
    RESULT_DIR
    / "md13_best_model"
)

best_model.save_pretrained(
    best_model_dir
)

tokenizer.save_pretrained(
    best_model_dir
)

archive_path = (
    shutil.make_archive(
        str(
            RESULT_DIR
            / "md13_best_model"
        ),
        "gztar",
        root_dir=best_model_dir,
    )
)

print(
    "\n"
    + "=" * 78
)
print(
    "MD13 COMPLETE"
)
print(
    "=" * 78
)

print(
    "Best epoch:",
    best_result["epoch"],
)
print(
    "Best accuracy:",
    best_result["accuracy"],
)
print(
    "Best Macro-F1:",
    best_result["macro_f1"],
)
print(
    "Summary:",
    summary_path,
)
print(
    "Best model archive:",
    archive_path,
)

print(
    "\nDecision references:"
)
print(
    "MD6 Epoch 2 accuracy = "
    "0.5016216216216216"
)
print(
    "MD6 Epoch 2 Macro-F1 = "
    "0.487684572655523"
)
print(
    "MD11 Epoch 3 Macro-F1 = "
    "0.48428708496304046"
)
print(
    "MD12 Epoch 3 Macro-F1 = "
    "0.4817452965264059"
)



------------------------------------------------------------------------------
EPOCH 1: /kaggle/working/md13_debertav3_base_weighted_alpha075/checkpoint-932
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Accuracy: 0.4481081081081081
Macro-F1: 0.39651834092334104
<=512 Macro-F1: 0.3640297660744545
>512 Macro-F1: 0.17086879484125833
Per-class F1: {'generic': 0.5616438356164384, 'probing': 0.18214285714285713, 'focus': 0.5379388448471121, 'telling': 0.30434782608695654}

------------------------------------------------------------------------------
EPOCH 2: /kaggle/working/md13_debertav3_base_weighted_alpha075/checkpoint-1864
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Accuracy: 0.3956756756756757
Macro-F1: 0.4073694539009622
<=512 Macro-F1: 0.4197740209444174
>512 Macro-F1: 0.1353615219099013
Per-class F1: {'generic': 0.5809018567639257, 'probing': 0.36622807017543857, 'focus': 0.2947976878612717, 'telling': 0.38755020080321284}

------------------------------------------------------------------------------
EPOCH 3: /kaggle/working/md13_debertav3_base_weighted_alpha075/checkpoint-2796
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Accuracy: 0.4221621621621622
Macro-F1: 0.4257923789001138
<=512 Macro-F1: 0.42876221163418393
>512 Macro-F1: 0.17760113449768622
Per-class F1: {'generic': 0.5763612217795485, 'probing': 0.3313032886723508, 'focus': 0.41786525359576077, 'telling': 0.37763975155279506}


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MD13 COMPLETE
Best epoch: 3
Best accuracy: 0.4221621621621622
Best Macro-F1: 0.4257923789001138
Summary: /kaggle/working/md13_results/md13_summary.json
Best model archive: /kaggle/working/md13_results/md13_best_model.tar.gz

Decision references:
MD6 Epoch 2 accuracy = 0.5016216216216216
MD6 Epoch 2 Macro-F1 = 0.487684572655523
MD11 Epoch 3 Macro-F1 = 0.48428708496304046
MD12 Epoch 3 Macro-F1 = 0.4817452965264059
